In [1]:
# If needed, install dependencies first:
# !pip install fredapi pandas pyarrow

import os
from datetime import date

import pandas as pd
from fredapi import Fred

# Read API key from environment variable
fred_api_key = os.environ["FRED_API_KEY"]

fred = Fred(api_key=fred_api_key)

# Pull monthly CPI data from FRED
cpi = fred.get_series(
    "CPIAUCSL",
    observation_start="2000-01-01",
    observation_end=date.today().strftime("%Y-%m-%d")
)

# Convert Series to DataFrame
df = cpi.reset_index()
df.columns = ["DATE", "CPI"]

# Format DATE like standard FRED data
df["DATE"] = pd.to_datetime(df["DATE"]).dt.strftime("%Y-%m-%d")

# Compute year-over-year percentage change
# CPIAUCSL is monthly, so 12 periods = 12 months
df["inflation_yoy"] = df["CPI"].pct_change(periods=12) * 100

# Classify inflation regimes
def classify_inflation(inflation):
    if pd.isna(inflation):
        return pd.NA
    elif inflation < 2:
        return "low"
    elif inflation <= 4:
        return "moderate"
    else:
        return "high"

df["inflation_regime"] = df["inflation_yoy"].apply(classify_inflation)

# Drop rows where YoY inflation cannot be calculated yet
df = df.dropna(subset=["inflation_yoy"])

# Create output folder and save as Parquet
output_path = "data/raw/cpi_inflation_regimes.parquet"
os.makedirs("data/raw", exist_ok=True)

df.to_parquet(output_path, index=False)

df.head()


,DATE,CPI,inflation_yoy,inflation_regime
12,2001-01-01,175.6,3.721205,moderate
13,2001-02-01,176.0,3.529412,moderate
14,2001-03-01,176.1,2.982456,moderate
15,2001-04-01,176.4,3.218256,moderate
16,2001-05-01,177.3,3.563084,moderate


In [2]:
pd.read_parquet("data/raw/cpi_inflation_regimes.parquet").head()


,DATE,CPI,inflation_yoy,inflation_regime
0,2001-01-01,175.6,3.721205,moderate
1,2001-02-01,176.0,3.529412,moderate
2,2001-03-01,176.1,2.982456,moderate
3,2001-04-01,176.4,3.218256,moderate
4,2001-05-01,177.3,3.563084,moderate
